## Gene panel statistics 

In [1]:
import pandas as pd
import numpy as np
import os
import anndata as ad
import squidpy as sq
import scanpy as sc
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

In [9]:
print(os.getcwd())
this_path = os.getcwd()

C:\Users\Chenxi Zhou\Desktop\Neural Crest\SingleCrest\merscope\result\analysis\2025MAY\Tutorial_Github\tutorial


In [2]:
adata = ad.read_h5ad("MER02_R1_filtered_stage22_subset.h5ad")
adata.X = adata.layers["raw_counts"]

path_save = str(os.getcwd()) + "//gene_panel_QC//"
Path.mkdir(Path(path_save), parents=True, exist_ok = True)


## statistics of gene expression

In [4]:

## statistics of gene expression matrix
adata.X = adata.layers["raw_counts"]
stat = sc.pp.calculate_qc_metrics(adata,percent_top=(1,5,10,15))
stat[0].to_csv(path_save + "cell_stat.csv")  
stat[1].to_csv(path_save + "gene_stat.csv")

stat[1]

,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts
ACTC1.L,3350,43.095860,3.786366,49.744974,287277.0,12.568206
CD44.L,2294,1.222022,0.798418,65.586559,8146.0,9.005405
CDH1.S,4113,9.354486,2.337420,38.298830,62357.0,11.040648
CDX4.L,990,0.292529,0.256601,85.148515,1950.0,7.576097
CHRD.1.S,3311,4.406841,1.687665,50.330033,29376.0,10.287968
DLX2.L,1201,1.174017,0.776577,81.983198,7826.0,8.965335
DLX3.L,4173,2.525352,1.259980,37.398740,16834.0,9.731215
EGR2.L,1522,0.678818,0.518090,77.167717,4525.0,8.417594
ELAVL3.L,1460,1.717072,0.999555,78.097810,11446.0,9.345483
EPHA2.L,4778,6.428143,2.005276,28.322832,42850.0,10.665484


In [6]:
## frequency of single transcript event (where only one transcript of a gene is detected in a cell)

from scipy.sparse import issparse

expr_matrix = adata[:, adata.var_names].X

if issparse(expr_matrix):
    expr_matrix = expr_matrix.toarray()

# create gene_by_cell expression dataFrame
gene_df = pd.DataFrame(
    expr_matrix,
    columns=adata.var_names,
    index=adata.obs_names
)


# count the frequency of single transcript event
gene_l = np.where(gene_df.to_numpy() == 1, 1,0).sum(axis=0) 

count_1 = pd.DataFrame(gene_l, 
                       index = adata.var_names,
                       columns=["number of single_transcript_event"])
count_1.to_csv(path_save + "single_transcript_event.csv")

count_1

,number of single_transcript_event
ACTC1.L,1296
CD44.L,1204
CDH1.S,1071
CDX4.L,738
CHRD.1.S,1092
DLX2.L,546
DLX3.L,1607
EGR2.L,1132
ELAVL3.L,836
EPHA2.L,1396


## Evaluate the spatial specificity

In [7]:
## Evaluate the spatial specificity  by Morans'I

# normalization and log1 transform of the data
# normalization by volume
adata.obs["volume"]  
adata.obs["volume_scaled"]=adata.obs["volume"]/1000   # scaled the volume by 1000 so the normalized count will above 1
adata.obs["volume_scaled"]

if (adata.obs["volume"] <= 0).any():
    raise ValueError("Volume values must be positive and non-zero!")  # Check if volume values are valid (non-zero to avoid division by zero)
    
adata.layers["normal_volume"] = adata.layers["raw_counts_bg_removeby1"] / adata.obs["volume_scaled"].values[:, None]

## log transformation

adata.layers["log1p_normal_volume"]=adata.layers["normal_volume"].copy()
sc.pp.log1p(adata, base=2, copy=False, chunked=None, chunk_size=None, layer="log1p_normal_volume", obsm=None)

# calculate the Moran's I 

adata.X = adata.layers["log1p_normal_volume"]

sq.gr.spatial_neighbors(adata, coord_type="generic")
sq.gr.spatial_autocorr(adata, mode="moran")
adata.uns["moranI"].to_csv(path_save + "moranI.csv")

adata.uns["moranI"]


,I,pval_norm,var_norm,pval_norm_fdr_bh
MYL1.L,0.931775,0.000000e+00,0.000047,0.000000e+00
ACTC1.L,0.924622,0.000000e+00,0.000047,0.000000e+00
MYOD1.S,0.895542,0.000000e+00,0.000047,0.000000e+00
PITX1.L,0.889972,0.000000e+00,0.000047,0.000000e+00
RAX.L,0.850988,0.000000e+00,0.000047,0.000000e+00
SOX3.S,0.848063,0.000000e+00,0.000047,0.000000e+00
SOX2.L,0.843485,0.000000e+00,0.000047,0.000000e+00
SOX10.L,0.818424,0.000000e+00,0.000047,0.000000e+00
HOXB9.S,0.785770,0.000000e+00,0.000047,0.000000e+00
SOX8.L,0.783968,0.000000e+00,0.000047,0.000000e+00


In [8]:
## draw normalized gene expression on sections

fig_save = Path(path_save + "\\spatial_expression\\")
Path(fig_save).mkdir(parents=True,exist_ok=True)
for gene in adata.var_names:
    fig, ax = plt.subplots( figsize=(12*1, 6*1) )
    sq.pl.spatial_scatter(
        adata,
        color=gene,
        cmap="gray_r",       # optional: change to 'viridis', 'plasma', etc.
        size=10,
        #return_fig=True,
        ax = ax,
        #show=False,
        shape = None,          # prevent pop-up
    )
    
    # Save to file
    plt.savefig(str(fig_save) + "\\" +  f"marker_{gene}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)  # Close to save memory